# 02 - Experiment Tracking with MLflow

**Goal:** Track training experiments (hyperparameters, metrics, models) so you can compare and reproduce them.

---

## The Problem

Your production training script logs to TensorBoard:

```python
# From training_semantic.py
writer = SummaryWriter(run_dir / training_name)
writer.add_scalar("train_loss", loss, global_step)
```

TensorBoard shows curves, but:
- Which hyperparameters produced the best model?
- What was the exact config for run `v8` vs `v7`?
- Where is the model file for the best run?

**MLflow** answers all of these in one place.

## What MLflow Tracks

```
MLflow Run
├── Parameters      ← what you configured (lr=0.001, batch_size=8)
├── Metrics         ← what happened (loss=0.34, dice=0.89)
├── Artifacts       ← files produced (model.pth, config.yaml, plots)
├── Tags            ← metadata (author, dataset_version, task)
└── Source           ← which code ran (git commit)
```

Multiple runs → compare them → pick the best → register it.

In [ ]:
import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os
import shutil
import yaml

# Use a temp directory for MLflow tracking
TRACKING_DIR = '/tmp/mlflow-demo'
if os.path.exists(TRACKING_DIR):
    shutil.rmtree(TRACKING_DIR)

mlflow.set_tracking_uri(f"file://{TRACKING_DIR}")
print(f"MLflow tracking: {TRACKING_DIR}")

## A Simple Training Example

Let's build a small training loop (similar to Phase 2 U-Net) and track it with MLflow.

In [ ]:
# Simple U-Net-like model for segmentation (from Phase 2)
class MiniUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=4, features=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, features, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(features, features * 2, 3, padding=1),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(features * 2, features, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(features, out_channels, 1),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


# Synthetic dataset: random images with simple shape masks
def generate_batch(batch_size=8, size=64, num_classes=4):
    images = torch.randn(batch_size, 1, size, size)
    labels = torch.randint(0, num_classes, (batch_size, size, size))
    return images, labels


print("Model and data ready.")

## Your First MLflow Run

The pattern is simple:

```python
with mlflow.start_run():
    mlflow.log_param("lr", 0.001)       # what you configured
    mlflow.log_metric("loss", 0.34)      # what happened
    mlflow.log_artifact("model.pth")     # what was produced
```

In [ ]:
def train_with_mlflow(config, num_epochs=20):
    """Train a model and log everything to MLflow."""
    
    with mlflow.start_run(run_name=config.get("run_name", None)):
        
        # --- Log parameters (what we configured) ---
        mlflow.log_param("learning_rate", config["lr"])
        mlflow.log_param("batch_size", config["batch_size"])
        mlflow.log_param("features", config["features"])
        mlflow.log_param("num_epochs", num_epochs)
        mlflow.log_param("optimizer", config["optimizer"])
        
        # Tags for organization
        mlflow.set_tag("task", "spine_semantic")
        mlflow.set_tag("dataset_version", config.get("dataset_version", "v1"))
        
        # --- Setup model and training ---
        model = MiniUNet(features=config["features"])
        criterion = nn.CrossEntropyLoss()
        
        if config["optimizer"] == "adam":
            optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
        else:
            optimizer = torch.optim.SGD(model.parameters(), lr=config["lr"])
        
        # --- Training loop ---
        train_losses = []
        for epoch in range(num_epochs):
            model.train()
            images, labels = generate_batch(batch_size=config["batch_size"])
            
            output = model(images)
            loss = criterion(output, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Log metric at each step
            mlflow.log_metric("train_loss", loss.item(), step=epoch)
            train_losses.append(loss.item())
        
        # --- Log final metrics ---
        final_loss = train_losses[-1]
        mlflow.log_metric("final_loss", final_loss)
        
        # --- Log artifacts (files) ---
        
        # Save and log model
        model_path = "/tmp/model.pth"
        torch.save(model.state_dict(), model_path)
        mlflow.log_artifact(model_path)
        
        # Save and log config
        config_path = "/tmp/config.yaml"
        with open(config_path, 'w') as f:
            yaml.dump(config, f)
        mlflow.log_artifact(config_path)
        
        # Save and log a training plot
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(train_losses)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title(f'Training Loss (lr={config["lr"]})')
        plot_path = "/tmp/training_curve.png"
        fig.savefig(plot_path)
        plt.close(fig)
        mlflow.log_artifact(plot_path)
        
        run_id = mlflow.active_run().info.run_id
        print(f"Run {run_id[:8]} | loss={final_loss:.4f} | lr={config['lr']} | features={config['features']}")
        
        return run_id

In [ ]:
# Create an experiment (groups related runs)
mlflow.set_experiment("spine-segmentation")

# Run 1: baseline
run1_id = train_with_mlflow({
    "run_name": "baseline",
    "lr": 0.001,
    "batch_size": 8,
    "features": 16,
    "optimizer": "adam",
    "dataset_version": "v1",
})

In [ ]:
# Run 2: higher learning rate
run2_id = train_with_mlflow({
    "run_name": "higher-lr",
    "lr": 0.01,
    "batch_size": 8,
    "features": 16,
    "optimizer": "adam",
    "dataset_version": "v1",
})

# Run 3: bigger model
run3_id = train_with_mlflow({
    "run_name": "bigger-model",
    "lr": 0.001,
    "batch_size": 4,
    "features": 32,
    "optimizer": "adam",
    "dataset_version": "v1",
})

# Run 4: SGD optimizer
run4_id = train_with_mlflow({
    "run_name": "sgd-optimizer",
    "lr": 0.01,
    "batch_size": 8,
    "features": 16,
    "optimizer": "sgd",
    "dataset_version": "v1",
})

## Querying Runs

MLflow stores everything in a structured way. You can query it programmatically.

In [ ]:
import pandas as pd

# Get all runs from our experiment
experiment = mlflow.get_experiment_by_name("spine-segmentation")
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.final_loss ASC"],  # best first
)

# Show comparison
comparison = runs[[
    "run_id",
    "tags.mlflow.runName",
    "params.learning_rate",
    "params.features",
    "params.optimizer",
    "metrics.final_loss",
]].copy()
comparison.columns = ["run_id", "name", "lr", "features", "optimizer", "final_loss"]
comparison["run_id"] = comparison["run_id"].str[:8]
print(comparison.to_string(index=False))

In [ ]:
# Find the best run
best_run = runs.iloc[0]
print(f"Best run: {best_run['tags.mlflow.runName']}")
print(f"  Loss: {best_run['metrics.final_loss']:.4f}")
print(f"  LR: {best_run['params.learning_rate']}")
print(f"  Features: {best_run['params.features']}")
print(f"  Run ID: {best_run['run_id'][:8]}")

## Loading a Run's Artifacts

You can retrieve any file that was logged with a run.

In [ ]:
# List artifacts from best run
client = mlflow.tracking.MlflowClient()
artifacts = client.list_artifacts(best_run['run_id'])
print("Artifacts:")
for a in artifacts:
    print(f"  {a.path} ({a.file_size} bytes)")

In [ ]:
# Download and load the config from the best run
artifact_path = mlflow.artifacts.download_artifacts(
    run_id=best_run['run_id'],
    artifact_path="config.yaml"
)
with open(artifact_path) as f:
    loaded_config = yaml.safe_load(f)

print("Config from best run:")
for k, v in loaded_config.items():
    print(f"  {k}: {v}")

In [ ]:
# Load the model from the best run
model_path = mlflow.artifacts.download_artifacts(
    run_id=best_run['run_id'],
    artifact_path="model.pth"
)

model = MiniUNet(features=int(best_run['params.features']))
model.load_state_dict(torch.load(model_path, weights_only=True))
print(f"Loaded model from run {best_run['run_id'][:8]}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## MLflow UI

MLflow has a web UI for visual comparison. In your Docker container:

```bash
mlflow ui --backend-store-uri file:///tmp/mlflow-demo --port 5000
```

Then open `http://localhost:5000` in your browser.

The UI shows:
- All runs in a table (sortable, filterable)
- Side-by-side metric comparison
- Training curves overlaid
- Artifact browser (download models, configs, plots)

## Connection to Your Production Code

Your current training script (`training_semantic.py`) uses TensorBoard:

```python
# Current: TensorBoard only
writer = SummaryWriter(run_dir / training_name)
writer.add_scalar("train_loss", loss.item(), global_step)
```

To add MLflow alongside TensorBoard:

```python
# Add MLflow tracking
import mlflow

mlflow.set_experiment("spine-segmentation")

with mlflow.start_run(run_name=training_name):
    # Log all config parameters
    mlflow.log_params({
        "task": config["task"]["name"],
        "version": config["task"]["version"],
        "learning_rate": config["training"]["optimizer"]["learning_rate"],
        "batch_size": config["training"]["batch_size"],
        "max_iterations": config["training"]["max_iterations"],
        "dataset": config["dataset"]["name"],
        "split": config["dataset"]["split_name"],
    })
    
    # Log config YAML as artifact
    mlflow.log_artifact(config_path)
    
    # Inside training loop:
    mlflow.log_metric("train_loss", loss.item(), step=global_step)
    
    # After validation:
    mlflow.log_metric("val_dice_mean", mean_dice, step=global_step)
    for class_name, dice_val in per_class_dice.items():
        mlflow.log_metric(f"val_dice_{class_name}", dice_val, step=global_step)
    
    # Save best model as artifact
    mlflow.log_artifact(best_model_path)
```

**What this gives you:**

| Question | TensorBoard | MLflow |
|----------|-------------|--------|
| See training curves | Yes | Yes |
| Compare runs side-by-side | Limited | Yes |
| What config produced model v8? | Check files manually | `mlflow.search_runs()` |
| Download best model | Find file path | `mlflow.artifacts.download_artifacts()` |
| Which dataset version? | Not tracked | Logged as parameter |
| FDA audit: full lineage | Impossible | Query API |

## Summary

| MLflow Concept | What it does | Example |
|----------------|-------------|--------|
| **Experiment** | Groups related runs | `spine-segmentation` |
| **Run** | One training session | `baseline`, `higher-lr` |
| **Parameter** | Input configuration | `lr=0.001`, `batch_size=8` |
| **Metric** | Output measurement | `loss=0.34`, `dice=0.89` |
| **Artifact** | Output file | `model.pth`, `config.yaml` |
| **Tag** | Metadata label | `dataset_version=v1` |

**Key takeaway:** MLflow makes experiments searchable and reproducible. Instead of "which folder had the good model?", you ask MLflow.

**Next:** Model Registry — promoting the best model to production.